# Simplest Sequential Workflow

**A hands-on guide to building sequential workflows with Microsoft Agent Framework**


📖 **Tutorial Reference:** [Simple Sequential Workflow - Microsoft Learn](https://learn.microsoft.com/en-us/agent-framework/tutorials/workflows/simple-sequential-workflow?pivots=programming-language-python)

---

## What You'll Learn

This notebook demonstrates the **simplest form of sequential workflows** - the foundation for building complex AI Agent systems. You'll learn:

- 🧩 What is an **Executor** and how to define them
- 🔗 How to connect executors using **Edges**
- 🔄 How to build a **Workflow** using WorkflowBuilder
- 📤 How to pass data between workflow steps
- 📊 How to stream events for real-time observability

---

## 1. Import Required Libraries

First, let's import the necessary components from the Microsoft Agent Framework:

In [1]:
import asyncio
from typing_extensions import Never
from agent_framework import WorkflowBuilder, WorkflowContext, WorkflowOutputEvent, Executor, handler, executor

---

## 2. Core Concepts

### 🧩 What is an Executor?

**Executors** are the fundamental building blocks of workflows. Think of them as:

- 🏗️ **Autonomous processing units** that receive typed messages
- ⚙️ **Workers** that perform specific operations
- 📡 **Message producers** that can emit output messages or events

**Two Ways to Define an Executor:**

1. **Class-based** - Create a custom class that inherits from `Executor` with methods decorated with `@handler`
2. **Function-based** - Use a standalone async function decorated with `@executor`

**Key Annotations:**
- `@handler` - Marks a method as a message handler (for class-based executors)
- `@executor(id="...")` - Decorator for function-based executors
- `WorkflowContext[T]` - Specifies the type of messages the handler will emit

### Example 1: Class-Based Executor

The `TrimSpace` executor receives a string, trims whitespace, and passes the result to the next executor.

**Key Points:**
- Inherits from `Executor` base class
- Requires a unique `id` for identification
- Uses `@handler` decorator for message processing
- Uses `ctx.send_message(result)` to pass data forward

In [2]:
class TrimSpace(Executor):
    def __init__(self, id: str):
        super().__init__(id=id)

    @handler
    async def to_trim_space(self, text: str, ctx: WorkflowContext[str]) -> None:
        """Trim the space in the sentence and forward it to the next node.

        Note: The WorkflowContext is parameterized with the type this handler will
        emit. Here WorkflowContext[str] means downstream nodes should expect str.
        """
        result = text.strip()

        # Send the result to the next executor in the workflow.
        await ctx.send_message(result)

### Example 2: Function-Based Executor

The `count_character` executor receives the trimmed string and counts its characters, yielding the final result.

**Key Points:**
- Uses `@executor(id="...")` decorator
- `WorkflowContext[Never, str]` indicates this executor yields output but doesn't send messages to other executors
- Uses `ctx.yield_output(result)` to produce the workflow's final output

In [3]:
@executor(id="count_character_executor")
async def count_character(text: str, ctx: WorkflowContext[Never, str]) -> None:
    """Count the input and yield the workflow output."""
    result = len(text)

    # Yield the final output for this workflow run
    await ctx.yield_output(result)

---

### 🔄 What is a Workflow?

A **Workflow** is a directed graph of executors connected by **edges** (data flow connections). It defines:

- 📍 **Start executor** - Where execution begins
- 🔗 **Edges** - Connections between executors showing data flow
- 🎯 **End points** - Executors that produce final output

**Building a Workflow with WorkflowBuilder:**

```python
workflow = (
    WorkflowBuilder()
    .add_edge(executor1, executor2)  # Connect executors
    .set_start_executor(executor1)   # Define starting point
    .build()                         # Create the workflow
)
```

## 3. Building the Workflow

Now let's connect our executors into a sequential workflow:

**Data Flow:**
```
Input Text → [TrimSpace] → Trimmed Text → [count_character] → Character Count
```

This creates a simple two-step pipeline where the output of the first executor becomes the input of the second.

In [4]:
trim_space = TrimSpace(id="trim_space_executor")

workflow = (
    WorkflowBuilder()
    .add_edge(trim_space, count_character)
    .set_start_executor(trim_space)
    .build()
)

---

### 📊 Event Streaming & Observability

One powerful feature of the framework is **event streaming**. When running a workflow, you can subscribe to real-time events that show:

- 🚀 `WorkflowStartedEvent` - Workflow execution begins
- ⏳ `WorkflowStatusEvent` - Status changes (IN_PROGRESS, IDLE, etc.)
- ▶️ `ExecutorInvokedEvent` - An executor starts processing
- ✅ `ExecutorCompletedEvent` - An executor finishes
- 🎯 `WorkflowOutputEvent` - Final output is produced

This enables real-time monitoring and debugging of your workflows.

## 4. Running the Workflow

Let's execute our workflow with the input text **"This is my first Workflow"**

**Expected Flow:**
1. `TrimSpace` receives "This is my first Workflow"
2. It trims spaces (result: same string - no leading/trailing spaces)
3. `count_character` receives the trimmed text
4. It counts characters (25 characters total)
5. Workflow outputs: **25**

In [5]:
async for event in workflow.run_stream("This is my first Workflow"):
    print(f"Event: {event}")
    if isinstance(event, WorkflowOutputEvent):
        print(f"Workflow completed with result: {event.data}")

Event: WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)
Event: WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)
Event: ExecutorInvokedEvent(executor_id=trim_space_executor, data=This is my first Workflow)
Event: ExecutorCompletedEvent(executor_id=trim_space_executor, data=['This is my first Workflow'])
Event: SuperStepStartedEvent(iteration=1, data=None)
Event: ExecutorInvokedEvent(executor_id=count_character_executor, data=This is my first Workflow)
Event: WorkflowOutputEvent(data=25, executor_id=count_character_executor)
Workflow completed with result: 25
Event: ExecutorCompletedEvent(executor_id=count_character_executor, data=[25])
Event: SuperStepCompletedEvent(iteration=1, data=None)
Event: WorkflowStatusEvent(state=WorkflowRunState.IDLE, data=None, origin=WorkflowEventSource.FRAMEWORK)


---

## 🎓 Summary & Key Takeaways

Congratulations! You've built your first sequential workflow with Microsoft Agent Framework.

### What We Built

A **2-step sequential workflow** that:
1. ✅ Trims whitespace from input text
2. 🔢 Counts the characters in the processed text

### Core Concepts Mastered

| Concept | Description | How We Used It |
|---------|-------------|----------------|
| **Executor** | Processing unit that handles messages | `TrimSpace` class and `count_character` function |
| **Handler** | Decorator for message processing methods | `@handler` on `to_trim_space()` |
| **@executor** | Decorator for function-based executors | `@executor(id="count_character_executor")` |
| **Workflow** | Directed graph of connected executors | Built with `WorkflowBuilder()` |
| **Edges** | Connections between executors | `.add_edge(trim_space, count_character)` |
| **Context** | Workflow execution context with typing | `WorkflowContext[str]` |
| **send_message** | Pass data to next executor | `await ctx.send_message(result)` |
| **yield_output** | Produce final workflow output | `await ctx.yield_output(result)` |

### Next Steps

Ready to build more complex workflows? Try:
- 🔄 Adding conditional branching
- 🌐 Integrating with LLMs and external APIs
- 🧠 Building multi-agent systems
- 📈 Adding custom event handlers

---

**🔗 Links & Resources**
- 📖 [Microsoft Learn Tutorial](https://learn.microsoft.com/en-us/agent-framework/tutorials/workflows/simple-sequential-workflow?pivots=programming-language-python)
- 💻 [Microsoft Agent Framework](https://github.com/microsoft/agent-framework)

**Happy Workflow Building!** 🚀

In [6]:
from agent_framework import WorkflowBuilder, WorkflowViz
viz = WorkflowViz(workflow)

In [ ]:
!pip install graphviz

In [ ]:
print(viz.save_png("workflow.png"))